# Setup — MECTESIS en Vertex AI Workbench

Ejecutar cada celda en orden. Solo la primera vez en una instancia nueva.

**Repositorio:** https://github.com/DavidGuzzi/MECTESIS  
**Tiempo estimado:** ~5-8 min total

## 1 — Clonar o actualizar el repositorio

In [ ]:
import os
from pathlib import Path

REPO = Path.home() / "MECTESIS"

if not REPO.exists():
    !git clone https://github.com/DavidGuzzi/MECTESIS.git {REPO}
    print(f"Clonado en {REPO}")
else:
    !git -C {REPO} pull
    print(f"Repo actualizado: {REPO}")

## 2 — Pararse en la raíz del repo

In [ ]:
%cd ~/MECTESIS
!pwd

## 3 — Verificar / reparar torch

Vertex AI tiene torch pre-instalado. Si está roto (común después de un `pip install`), lo reinstalamos.

In [ ]:
try:
    import torch
    _ = torch.zeros(1)  # fuerza carga de libs nativas
    print(f"torch {torch.__version__} OK — device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
except (OSError, ImportError) as e:
    import subprocess, sys
    print(f"torch roto: {e}")
    print("Reinstalando torch CPU (--no-deps, puede tardar ~2 min)...")
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install",
         "--force-reinstall", "--no-deps", "torch",
         "--index-url", "https://download.pytorch.org/whl/cpu", "--quiet"],
    )
    if r.returncode != 0:
        print("Reintentando con --user...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install",
             "--user", "--no-deps", "torch",
             "--index-url", "https://download.pytorch.org/whl/cpu", "--quiet"],
            check=True,
        )
    from IPython.display import display, HTML
    display(HTML(
        "<div style='border:2px solid red;padding:12px;background:#fff3f3'>"
        "<b style='color:red'>&#9888; torch reinstalado.</b><br>"
        "El kernel tiene la version rota en cache. "
        "<b>Reinicia el kernel ahora:</b> Kernel → Restart Kernel<br>"
        "Despues, vuelve a ejecutar desde la celda 3."
        "</div>"
    ))
    raise SystemExit("Reinicia el kernel y vuelve a ejecutar desde la celda 3.")

## 4 — Instalar dependencias de mectesis

In [ ]:
!pip install -e . --quiet
print("Instalación completada")

## 5 — Verificar imports

In [ ]:
from mectesis.dgp import ARpDGP, MAqDGP, ARMApqDGP, RandomWalk
from mectesis.models.arima import ARIMAModel
from mectesis.simulation import MonteCarloEngine
from mectesis.metrics import BiasVarianceMSE
print("mectesis importado correctamente")

## 6 — Cargar Chronos-2

La primera vez descarga ~1.5 GB desde Hugging Face. Las siguientes usa caché.

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

from mectesis.models import ChronosModel
chronos = ChronosModel(device=device)
print("Chronos-2 listo")

## 7 — Crear directorio de resultados

In [ ]:
from pathlib import Path
Path("results/univariate_v3").mkdir(parents=True, exist_ok=True)
print("Directorio listo:", Path("results/univariate_v3").resolve())

## 8 — Smoke test (opcional)

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from mectesis.dgp import ARpDGP
from mectesis.models.arima import ARIMAModel
from mectesis.simulation import MonteCarloEngine

dgp = ARpDGP(phis=[0.5], sigma=1.0, seed=42)
engine = MonteCarloEngine(dgp, [ARIMAModel((1, 0, 0))], seed=42)
res = engine.run_monte_carlo(n_sim=5, T=50, horizon=6, dgp_params={}, verbose=True)
print("RMSE h=1:", round(float(res["ARIMA(1, 0, 0)"]["rmse"].iloc[0]), 4))
print("Smoke test OK")

---
## Listo para ejecutar el notebook principal

Abrir **`notebooks/experimentos_univariados_v3_cloud.ipynb`** y hacer:  

**Kernel → Restart Kernel and Run All Cells**

Los resultados se guardan en `results/univariate_v3/` (CSV por experimento + `.log`).